In [ ]:
import base64
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/FaiazPronoy/fabrid-app.git"
REPO = Path("/kaggle/working/fabrid-app")
os.environ["PYTHONUNBUFFERED"] = "1"

token = ""
try:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN") or ""
except Exception as error:
    print(f"secret GITHUB_TOKEN not available ({type(error).__name__}), cloning without it")

auth, hidden = [], []
if token:
    encoded = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {encoded}"]
    hidden = [token, encoded]
if REPO.exists():
    shutil.rmtree(REPO)
clone = ["git", *auth, "clone", "--quiet", "--depth", "1", REPO_URL, str(REPO)]
result = subprocess.run(clone, capture_output=True, text=True)
auth, token = [], ""
if result.returncode != 0:
    message = result.stderr
    for secret in hidden:
        message = message.replace(secret, "***")
    raise RuntimeError(f"git clone failed: {message}")
hidden = []
print(f"cloned {REPO_URL}", flush=True)

launch = [sys.executable, "-u", str(REPO / "kaggle" / "launch.py")]
process = subprocess.Popen(
    launch, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
for line in process.stdout:
    print(line, end="", flush=True)
code = process.wait()
raise RuntimeError(f"the demo launcher stopped with code {code}")